# A1.12 · Proving guardrails work

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.11 · Building outcome-driven guardrails, layer by layer](https://spbreed.github.io/cyber-commons/lessons/A1.11.html)**.

| | |
|---|---|
| Open-source tooling | garak, promptfoo |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A guardrail claim is not evidence. "The agent will not exfiltrate data" is a
sentence; what an auditor and a red team both want is the measurement behind it.

Two things make that measurement different from ordinary testing.

**The attacker retries.** A control that holds 97 times in 100 sounds strong and
is not, because the attacker is not sampling randomly — they run it again. At a
3% per-attempt success rate, the expected number of attempts before a success is
about 33, and the probability of at least one success within 100 attempts is
over 95%. For an attacker paying fractions of a cent per attempt, a 97% control
is a speed bump with a number attached.

**The system is stochastic.** The same input can produce a refusal and a
compliance on two runs. So a single passing test tells you almost nothing — the
question is not "did it hold" but "at what rate, with what confidence interval,
and how many attempts does an attacker need".

The output of this lesson is a number you can defend: **the attacker's expected
cost per success.** That converts a guardrail from an assertion into an economic
statement, which is the form a risk committee can actually act on.

## 2 · The arithmetic of a control that mostly holds\n\nPer-attempt success rate, and what it means to someone who can retry.

In [ ]:
def attempts_for(p_success, confidence=0.95):
    """How many attempts before the attacker succeeds, with `confidence`."""
    import math
    if p_success <= 0: return float("inf")
    if p_success >= 1: return 1
    return math.ceil(math.log(1 - confidence) / math.log(1 - p_success))

print(f"{'guardrail holds':>16s}{'attacker p':>12s}{'expected tries':>16s}"
      f"{'tries for 95%':>15s}")
for hold in (0.50, 0.90, 0.97, 0.99, 0.999, 0.9999):
    p = 1 - hold
    print(f"{hold:>15.2%} {p:>11.2%} {1/p:>15.1f} {attempts_for(p):>15d}")
print()
print("A 97% guardrail is defeated within 98 attempts, 19 times out of 20.")
print("At $0.002 an attempt that is about twenty cents.")

## 3 · Measure it, do not assert it\n\nA bypass suite against a guardrail with a deterministic, seeded stand-in for the model's variability.

In [ ]:
import random

def guardrail(payload, strength, rng):
    """Holds with probability `strength`. Deterministic given the seed."""
    return rng.random() < strength

def bypass_suite(strength, attempts=1000, seed=7):
    rng = random.Random(seed)          # seeded: the same suite every run
    held = sum(guardrail("payload", strength, rng) for _ in range(attempts))
    return {"attempts": attempts, "held": held, "bypassed": attempts - held,
            "hold_rate": held / attempts}

for strength in (0.97, 0.99, 0.999):
    r = bypass_suite(strength, attempts=5000)
    p = 1 - r["hold_rate"]
    cost = (1 / p * 0.002) if p else float("inf")
    print(f"claimed {strength:.1%}  measured {r['hold_rate']:.2%}  "
          f"bypassed {r['bypassed']:>3d}/{r['attempts']}  "
          f"attacker cost per success ${cost:,.2f}")
print()
print("The measured rate is what goes in the report. The claimed rate is what")
print("the vendor said.")

## 4 · Where it breaks — one run, and the confidence interval nobody quotes\n\nA single test is not a measurement of a stochastic system.

In [ ]:
def single_run(strength, seed):
    rng = random.Random(seed)
    return guardrail("payload", strength, rng)

results = [single_run(0.97, s) for s in range(20)]
print(f"twenty single-run tests of the same 97% guardrail:")
print("   " + " ".join("HOLD" if r else "FAIL" for r in results))
print(f"   -> {results.count(True)} passed, {results.count(False)} failed")
suite = bypass_suite(0.97, attempts=1000)
print(f"   the same control over {suite['attempts']} attempts: "
      f"{suite['bypassed']} bypasses")
print()
print("Twenty demos, twenty passes, and a control that fails about three times")
print("in a hundred. The demo was not lucky - it was too small to contain a")
print("failure, which is a different and much more comfortable kind of wrong.")

def wilson(held, n, z=1.96):
    """A confidence interval, because a rate without one is a rumour."""
    if n == 0: return (0.0, 1.0)
    p = held / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    m = z * ((p*(1-p)/n + z*z/(4*n*n)) ** 0.5) / d
    return (max(0.0, c - m), min(1.0, c + m))

for n in (10, 100, 1000):
    r = bypass_suite(0.97, attempts=n)
    lo, hi = wilson(r["held"], n)
    print(f"   n={n:>4}  measured {r['hold_rate']:.2%}  95% CI [{lo:.2%}, {hi:.2%}]")
print()
print("At n=10 the interval is wide enough to contain both 'strong control' and")
print("'no control'. The sample size is part of the claim.")
assert results.count(True) == 20 and suite["bypassed"] > 0

## 5 · The control — state the bar before you run the suite\n\nAn acceptance threshold decided after seeing the result is not a threshold.

In [ ]:
def accepts(measured_lo, required_hold, attacker_budget, cost_per_attempt=0.002):
    """Two tests: the lower CI bound clears the bar, and the attacker cannot
    afford the expected attempts at that bound."""
    p = 1 - measured_lo
    afford = attacker_budget / cost_per_attempt
    return {"clears_bar": measured_lo >= required_hold,
            "expected_attempts": (1/p) if p else float("inf"),
            "attacker_can_afford": (1/p if p else float("inf")) <= afford}

for hold, bar in ((0.97, 0.999), (0.999, 0.999), (0.99999, 0.999)):
    r = bypass_suite(hold, attempts=2000)
    lo, _ = wilson(r["held"], 2000)
    v = accepts(lo, bar, attacker_budget=100.0)
    print(f"claimed {hold:<9.5f} CI-low {lo:.3%}  clears {bar:.1%} bar: "
          f"{str(v['clears_bar']):5s}  attacker needs "
          f"{v['expected_attempts']:,.0f} tries, affordable: {v['attacker_can_afford']}")
print()
print("The bar is set by what the outcome is worth, not by what the control")
print("happens to score. A control that clears the bar and is still affordable")
print("to brute-force has not passed - it has been priced.")

## 6 · Verify — evidence a red team will accept

In [ ]:
r = bypass_suite(0.999, attempts=5000)
lo, hi = wilson(r["held"], 5000)
p = 1 - lo
evidence = {
  "control": "no mail leaves the corporate tenant",
  "layer": 7,
  "attempts": r["attempts"],
  "bypasses": r["bypassed"],
  "hold_rate": round(r["hold_rate"], 5),
  "ci95": [round(lo, 5), round(hi, 5)],
  "expected_attacker_attempts": round(1/p, 1) if p else None,
  "cost_per_attempt_usd": 0.002,
  "expected_cost_per_success_usd": round(1/p * 0.002, 2) if p else None,
  "suite_seed": 7,
}
for k, v in evidence.items():
    print(f"   {k:32s}{v}")
print()
print("Every field here is reproducible and every one is falsifiable. That is")
print("the difference between a guardrail claim and guardrail evidence.")
assert evidence["bypasses"] >= 0 and evidence["ci95"][0] <= evidence["hold_rate"]

## What you just proved

The retry arithmetic prints: a 97% guardrail is defeated within 98 attempts 19 times out of 20, for about twenty cents. Twenty single-run tests of the same control return a mix of passes and failures, and the Wilson interval at n=10 is wide enough to contain both 'strong control' and 'no control'. The lesson ends with a reproducible evidence record carrying a seed, an interval and an attacker cost per success.

## Your turn

Take a guardrail you rely on and estimate its per-attempt hold rate honestly. Then compute the attacker's cost per success at your own inference prices. Most teams discover the number is smaller than the coffee they bought while reading this.

---

**Next → [A2.1 · "Who is calling?"](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*